# Agente de Análise de Arquivos Potencialmente Maliciosos
### Componentes:
Clara Virgínia Nascimento Santos - 20240002212 \
Heloísa Maria Alves Severo - 20240057945

### 1. Import do csv com 10 casos e conversao para JSON

In [68]:
# imports
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"


import pandas as pd
import json
import agentkit
import inspect
from typing import Callable, get_type_hints
import torch
from agentkit import LLM

ERROR: Package 'agentkit' requires a different Python: 3.9.6 not in '>=3.11'
You should consider upgrading via the '/Users/kimitayo/Documents/UFRN/TEICA/unidade 1/projeto/venv/bin/python3 -m pip install --upgrade pip' command.


ModuleNotFoundError: No module named 'torch'

In [61]:
# leitura do casos.csv
df = pd.read_csv('casos.csv')
df.head(10)

,id,nome_arquivo,extensao,tamanho_kb,origem,hash,assinatura_digital,criou_processo,alterou_registro,criou_arquivo,conexao_externa,resultado_esperado
0,1,relatorio.pdf,.pdf,850,download,hash_benigno_001,valida,False,False,False,False,BENIGNO
1,2,invoice.exe,.exe,2450,email,hash_malicioso_001,invalida,True,True,False,True,MALICIOSO
2,3,foto.jpg,.jpg,3200,pendrive,hash_desconhecido_001,ausente,False,False,False,False,BENIGNO
3,4,update.exe,.exe,1800,site_desconhecido,hash_desconhecido_002,ausente,True,True,True,True,MALICIOSO
4,5,documento.docx,.docx,540,email,hash_benigno_002,ausente,False,False,False,False,BENIGNO
5,6,setup.exe,.exe,5200,site_oficial,hash_desconhecido_003,valida,True,False,False,True,SUSPEITO
6,7,script.ps1,.ps1,45,email,hash_desconhecido_004,ausente,True,True,False,True,MALICIOSO
7,8,imagem.png,.png,1200,download,hash_benigno_003,ausente,False,False,False,False,BENIGNO
8,9,game_crack.exe,.exe,7600,site_desconhecido,hash_desconhecido_005,ausente,True,False,True,True,MALICIOSO
9,10,backup.zip,.zip,15000,rede_interna,hash_desconhecido_006,ausente,False,False,False,False,SUSPEITO


In [62]:
# converter para dict e separar o resultado_separado
casos = []

for _, linha in df.iterrows(): # percorre o df linha por linha
    arquivo = linha.drop("resultado_esperado").to_dict() #forma o arquivo sem o resultado_esperado

    caso = { # é o caso completo
        "arquivo": arquivo,
        "resultado_esperado": linha["resultado_esperado"]
    }
    casos.append(caso)

len(casos)
casos[0]

{'arquivo': {'id': 1,
  'nome_arquivo': 'relatorio.pdf',
  'extensao': '.pdf',
  'tamanho_kb': 850,
  'origem': 'download',
  'hash': 'hash_benigno_001',
  'assinatura_digital': 'valida',
  'criou_processo': False,
  'alterou_registro': False,
  'criou_arquivo': False,
  'conexao_externa': False},
 'resultado_esperado': 'BENIGNO'}

In [63]:
# converter para json o arquivo a ser enviado pro agente
arquivo = casos[0]["arquivo"] # convertendo o primeiro caso sem o resultado_esperado para json
arquivo_json = json.dumps(
    arquivo,
    ensure_ascii=False,
    indent=4
    )

print(arquivo_json)

{
    "id": 1,
    "nome_arquivo": "relatorio.pdf",
    "extensao": ".pdf",
    "tamanho_kb": 850,
    "origem": "download",
    "hash": "hash_benigno_001",
    "assinatura_digital": "valida",
    "criou_processo": false,
    "alterou_registro": false,
    "criou_arquivo": false,
    "conexao_externa": false
}


### 2. Import do indicadores.csv

In [64]:
df_indicadores = pd.read_csv('indicadores.csv')

df_indicadores.head()

,hash,status,descricao
0,hash_benigno_001,benigno,Arquivo conhecido como legítimo e sem indícios...
1,hash_malicioso_001,malicioso,Arquivo previamente identificado como malicios...
2,hash_benigno_002,benigno,Arquivo conhecido como legítimo e sem indícios...
3,hash_benigno_003,benigno,Arquivo conhecido como legítimo e sem indícios...


In [65]:
# testar consulta manual
hash_consultado = "hash_malicioso_001"

resultado = df_indicadores[df_indicadores["hash"] == hash_consultado]
resultado

,hash,status,descricao
1,hash_malicioso_001,malicioso,Arquivo previamente identificado como malicios...


In [66]:
TYPE_NAMES = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean"
}


def tool(fn: Callable) -> Callable:
    """Anexa fn.tool_schema por introspecção e devolve a própria função."""
    hints = get_type_hints(fn)
    names = list(inspect.signature(fn).parameters)

    fn.tool_schema = {
        "name": fn.__name__,
        "description": inspect.getdoc(fn).splitlines()[0],
        "parameters": {
            name: {"type": TYPE_NAMES[hints[name]]}
            for name in names
        },
        "required": names,
    }

    return fn

### 3. Construcao das ferramentas

In [ ]:
# tool1 -> consultar indicadores conhecidos
@tool
def consultar_indicador(hash: str) -> str:
    resultado = df_indicadores[df_indicadores["hash"] == hash]
    if len(resultado) > 0:
        return resultado.to_dict(orient="records")[0] # retorna o primeiro registro encontrado
    else:
        return None

# tool2 -> consultar os metadados (nome, extensao, tamanho, origem, assinatura)
@tool
def analisar_metadados(
    nome_arquivo: str,
    extensao: str,
    tamanho_kb: int,
    origem: str,
    assinatura_digital: str
) -> str:

    return str({
        "nome_arquivo": nome_arquivo,
        "extensao": extensao,
        "tamanho_kb": tamanho_kb,
        "origem": origem,
        "assinatura_digital": assinatura_digital
    })

# tool3 -> acoes observadas durante a análises
@tool
def analisar_comportamento(
    criou_processo: bool,
    alterou_registro: bool,
    criou_arquivo: bool,
    conexao_externa: bool
) -> str:
    """Analisa as ações observadas durante a análise do arquivo."""

    return str({
        "criou_processo": criou_processo,
        "alterou_registro": alterou_registro,
        "criou_arquivo": criou_arquivo,
        "conexao_externa": conexao_externa
    })

AttributeError: 'NoneType' object has no attribute 'splitlines'

In [ ]:
consultar_indicador("hash_malicioso_001")

NameError: name 'consultar_indicador' is not defined